# Step 3: LoRA fine-tuning with augmented BIPIA clean controls

**Capstone: Prompt-Injection Defense Evaluation, Notebook 10c**

Question: NB10b failed because BIPIA has only 50 clean controls (35 in train). The model learned 'BIPIA email format = INJECTION' rather than discriminating attack content from clean content. Cohen d on BIPIA test was 0.13 (Variant B) — degenerate. Does adding more clean BIPIA-format examples fix this?

## What changed from NB10b

- Input CSV: `bipia_email_qa_prompts_augmented.csv` (1,050 rows) instead of `bipia_email_qa_prompts.csv` (800 rows)
- 50 BIPIA base emails each paired with 5 generic legitimate user questions = 250 new clean rows
- Total BIPIA dataset: 750 attacks + 300 clean (28.6% clean vs 6.3% before)
- Stratified 70/15/15 split now gives ~210 clean training rows vs 35 before
- Same LoRA recipe (ProtectAI base, r=16, alpha=32, all-linear, 3 epochs, class-weighted loss)

## Two variants again (same as NB10b)

| Variant | Training data | Question it answers |
|---|---|---|
| **A. BIPIA-augmented only** | Augmented BIPIA train split (~735 rows) | Is BIPIA learnable with sufficient clean controls? |
| **B. Combined** | eval_set train (~3,182) + BIPIA-augmented train (~735) = ~3,917 rows | Can a single LoRA cover both direct + indirect injection given enough clean controls for each? |

## Required uploads to Drive

From repo to `MyDrive/capstone_lora/data/`:
- **NEW: `bipia_email_qa_prompts_augmented.csv`** (1,050 rows, from `scripts/augment_bipia_clean.py`)

Already in place from NB08/NB10/NB10b:
- `eval_set.parquet`, `eval_set_splits.parquet`

Total wall time: ~10 min for both variants on L4.

## 1. Environment setup

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/capstone_lora')
DATA_DIR = DRIVE_ROOT / 'data'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SET_PATH = DATA_DIR / 'eval_set.parquet'
EVAL_SPLITS_PATH = DATA_DIR / 'eval_set_splits.parquet'
BIPIA_PROMPTS = DATA_DIR / 'bipia_email_qa_prompts_augmented.csv'
BIPIA_SPLITS_PATH = DATA_DIR / 'bipia_splits_augmented.parquet'

ADAPTER_DIR_A = DRIVE_ROOT / 'adapters' / 'lora_v3a_bipia_aug_only'
ADAPTER_DIR_B = DRIVE_ROOT / 'adapters' / 'lora_v3b_combined_aug'
ADAPTER_DIR_A.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR_B.mkdir(parents=True, exist_ok=True)

for p in [EVAL_SET_PATH, EVAL_SPLITS_PATH, BIPIA_PROMPTS]:
    print(f'  {p.name}: {"OK" if p.exists() else "MISSING upload first"}')

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '--quiet', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers>=4.53', 'datasets', 'accelerate', 'scikit-learn',
    'peft', 'tqdm', 'sentencepiece'])
print('Packages installed.')

In [ ]:
import json
import time
import gc
import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score, matthews_corrcoef, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    USE_BF16 = cc[0] >= 8
    USE_FP16 = not USE_BF16
    print(f'GPU: {torch.cuda.get_device_name(0)}, precision: {"bf16" if USE_BF16 else "fp16"}')
else:
    USE_BF16, USE_FP16 = False, False
    print('No GPU detected; training will be slow.')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Load augmented BIPIA + eval_set + create reproducible BIPIA split

Stratification key for BIPIA is `attack_category` (16 categories: 1 control + 15 attack categories). All 300 control rows fall in the 'control' stratum, attacks split by category.

In [ ]:
bipia = pd.read_csv(BIPIA_PROMPTS)
bipia = bipia.rename(columns={'full_prompt': 'prompt', 'is_attack': 'label'})
bipia['dataset'] = 'bipia'
print(f'BIPIA augmented total: {len(bipia)}')
print(f'Class balance: {bipia["label"].value_counts().to_dict()}')
print(f'Attack categories: {bipia["attack_category"].nunique()}')

if BIPIA_SPLITS_PATH.exists():
    bipia_splits = pd.read_parquet(BIPIA_SPLITS_PATH)
    print(f'Loaded existing augmented BIPIA splits from {BIPIA_SPLITS_PATH}')
else:
    train_val, test_ = train_test_split(
        bipia, test_size=0.15, random_state=SEED, stratify=bipia['attack_category']
    )
    train, val = train_test_split(
        train_val, test_size=0.15/0.85, random_state=SEED, stratify=train_val['attack_category']
    )
    train['split'] = 'train'
    val['split'] = 'val'
    test_['split'] = 'test'
    bipia_splits = pd.concat([train, val, test_], ignore_index=True)
    bipia_splits.to_parquet(BIPIA_SPLITS_PATH, index=False)
    print(f'Created augmented BIPIA splits, saved to {BIPIA_SPLITS_PATH}')

print('\nBIPIA split sizes:')
print(bipia_splits.groupby('split')['label'].agg(['count', 'sum']))
print('\nClean controls per split (key vs NB10b: train was 35, now should be ~210):')
print(bipia_splits.groupby('split').apply(lambda d: (d['label']==0).sum(), include_groups=False))

In [ ]:
eval_splits = pd.read_parquet(EVAL_SPLITS_PATH)
print(f'eval_set splits loaded: {len(eval_splits)} rows total')
print('Class balance per split:')
print(eval_splits.groupby('split')['label'].agg(['count', 'sum', 'mean']).round(3))

## 3. Helpers: metrics with imbalance-robust additions (macro F1, balanced accuracy, MCC)

Headline metrics now include macro F1 / balanced accuracy / MCC alongside binary F1, so the report no longer reads positive-class F1 in isolation on imbalanced slices.

In [ ]:
def wilson_ci(successes, n, alpha=0.05):
    if n == 0:
        return (0.0, 1.0)
    r = binomtest(int(successes), int(n))
    lo, hi = r.proportion_ci(confidence_level=1 - alpha, method='wilson')
    return float(lo), float(hi)

def per_class_metrics(y_true, y_pred, label_name=''):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    try:
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)
    except Exception:
        bal_acc, mcc = 0.0, 0.0
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    n_attack = int((y_true == 1).sum())
    n_clean = int((y_true == 0).sum())
    rec_ci = wilson_ci(tp, tp + fn) if (tp + fn) > 0 else (0.0, 0.0)
    prec_ci = wilson_ci(tp, tp + fp) if (tp + fp) > 0 else (0.0, 0.0)
    asr = 1.0 - (tp / max(n_attack, 1))
    far = fp / max(n_clean, 1)
    return {
        'label': label_name, 'n': len(y_true), 'n_pos': n_attack, 'n_neg': n_clean,
        'precision': float(p), 'precision_ci': prec_ci,
        'recall': float(r), 'recall_ci': rec_ci,
        'f1': float(f), 'macro_f1': float(macro_f1),
        'balanced_accuracy': float(bal_acc), 'mcc': float(mcc),
        'accuracy': float(acc),
        'asr': float(asr), 'far': float(far),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    }

def print_metrics_table(metrics_list, title):
    print(f'\n=== {title} ===')
    print(f'{"Slice":<20} {"n":>4} {"n+":>4} {"F1":>6} {"macF1":>6} {"balAcc":>7} {"MCC":>6} {"ASR":>6} {"FAR":>6}')
    for m in metrics_list:
        print(f'{m["label"]:<20} {m["n"]:>4} {m["n_pos"]:>4} {m["f1"]:>6.3f} {m["macro_f1"]:>6.3f} '
              f'{m["balanced_accuracy"]:>7.3f} {m["mcc"]:>6.3f} {m["asr"]:>6.3f} {m["far"]:>6.3f}')

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            cw = torch.tensor(self.class_weights, device=logits.device, dtype=logits.dtype)
            loss_fn = torch.nn.CrossEntropyLoss(weight=cw)
        else:
            loss_fn = torch.nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', pos_label=1, zero_division=0)
    return {'accuracy': accuracy_score(labels, preds), 'precision': p, 'recall': r, 'f1': f1,
            'macro_f1': f1_score(labels, preds, average='macro', zero_division=0)}

In [ ]:
BASE_MODEL = 'ProtectAI/deberta-v3-base-prompt-injection-v2'
MAX_LENGTH = 512

def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[['prompt', 'label']].rename(columns={'label': 'labels'}).reset_index(drop=True)
    )

def train_lora(train_df, val_df, run_label, adapter_save_path, num_epochs=3,
               learning_rate=2e-4, batch_size=16):
    print(f'\n=== Training {run_label} on {BASE_MODEL} ===')
    print(f'  train: {len(train_df)} rows, val: {len(val_df)} rows')
    print(f'  train class balance: {train_df["label"].value_counts().to_dict()}')

    classes = np.array([0, 1])
    weights = compute_class_weight('balanced', classes=classes, y=train_df['label'].values)
    print(f'  class weights: 0={weights[0]:.3f}, 1={weights[1]:.3f}')

    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2,
        id2label={0: 'BENIGN', 1: 'INJECTION'},
        label2id={'BENIGN': 0, 'INJECTION': 1},
        ignore_mismatched_sizes=True,
    )
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
        lora_dropout=0.1, target_modules='all-linear', bias='none',
    )
    mdl = get_peft_model(mdl, cfg)
    mdl.to(device)
    n_trainable = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in mdl.parameters())
    print(f'  Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)')

    def tok_fn(batch):
        return tok(batch['prompt'], truncation=True, max_length=MAX_LENGTH, padding=False)
    tr_ds = to_hf_dataset(train_df).map(tok_fn, batched=True, remove_columns=['prompt'])
    vl_ds = to_hf_dataset(val_df).map(tok_fn, batched=True, remove_columns=['prompt'])
    coll = DataCollatorWithPadding(tokenizer=tok, padding=True, pad_to_multiple_of=8)

    args = TrainingArguments(
        output_dir=f'/content/lora_{run_label}',
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type='linear',
        logging_steps=25,
        eval_strategy='epoch',
        save_strategy='epoch',
        fp16=USE_FP16, bf16=USE_BF16,
        seed=SEED, report_to='none',
        load_best_model_at_end=True,
        metric_for_best_model='eval_macro_f1',
        greater_is_better=True,
    )

    t0 = time.time()
    tr_obj = WeightedTrainer(
        model=mdl, args=args, train_dataset=tr_ds, eval_dataset=vl_ds,
        data_collator=coll, compute_metrics=compute_metrics_fn,
        class_weights=weights.tolist(),
    )
    tr_obj.train()
    elapsed = time.time() - t0
    print(f'  Trained in {elapsed/60:.1f} min')

    mdl.save_pretrained(adapter_save_path)
    tok.save_pretrained(adapter_save_path)
    print(f'  Adapter saved to {adapter_save_path}')
    return mdl, tok, tr_obj, elapsed

In [ ]:
def evaluate_on(trainer, tok, test_df, slice_label):
    def tok_fn(batch):
        return tok(batch['prompt'], truncation=True, max_length=MAX_LENGTH, padding=False)
    te_ds = to_hf_dataset(test_df).map(tok_fn, batched=True, remove_columns=['prompt'])
    preds_obj = trainer.predict(te_ds)
    logits = preds_obj.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    y_pred = (probs > 0.5).astype(int)
    y_true = test_df['label'].values
    return {
        'metrics': per_class_metrics(y_true, y_pred, slice_label),
        'y_pred': y_pred.tolist(),
        'y_score': probs.tolist(),
    }

def cleanup_model(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Variant A: BIPIA-augmented-only LoRA

Sanity check: does the model now learn BIPIA discrimination with 210 clean training rows vs the 35 it had in NB10b? Look at Cohen d > 1.5 and macro F1 > 0.8 as the success criteria, not binary F1.

In [ ]:
bipia_train = bipia_splits[bipia_splits['split'] == 'train']
bipia_val = bipia_splits[bipia_splits['split'] == 'val']
bipia_test = bipia_splits[bipia_splits['split'] == 'test']

model_a, tok_a, trainer_a, time_a = train_lora(
    train_df=bipia_train, val_df=bipia_val,
    run_label='v3a_bipia_aug_only',
    adapter_save_path=ADAPTER_DIR_A,
    num_epochs=3, learning_rate=2e-4, batch_size=16,
)

In [ ]:
eval_test = eval_splits[eval_splits['split'] == 'test'].reset_index(drop=True)

result_a_bipia = evaluate_on(trainer_a, tok_a, bipia_test, 'BIPIA-aug test')
result_a_evalset = evaluate_on(trainer_a, tok_a, eval_test, 'eval_set test')

metrics_a = [result_a_bipia['metrics'], result_a_evalset['metrics']]
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = eval_test[eval_test['dataset'] == ds].reset_index(drop=True)
    if len(sub) > 0:
        r = evaluate_on(trainer_a, tok_a, sub, f'  eval_set/{ds}')
        metrics_a.append(r['metrics'])

print_metrics_table(metrics_a, 'Variant A: BIPIA-augmented-only LoRA')

bipia_test_with_pred = bipia_test.reset_index(drop=True).copy()
bipia_test_with_pred['pred_a'] = result_a_bipia['y_pred']
bipia_test_with_pred['score_a'] = result_a_bipia['y_score']

In [ ]:
cleanup_model(model_a, trainer_a)
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 5. Variant B: combined (eval_set + BIPIA-augmented) LoRA

In [ ]:
eval_train = eval_splits[eval_splits['split'] == 'train'][['prompt', 'label', 'dataset']].copy()
eval_val = eval_splits[eval_splits['split'] == 'val'][['prompt', 'label', 'dataset']].copy()

bipia_train_min = bipia_train[['prompt', 'label']].copy()
bipia_train_min['dataset'] = 'bipia'
bipia_val_min = bipia_val[['prompt', 'label']].copy()
bipia_val_min['dataset'] = 'bipia'

combined_train = pd.concat([eval_train, bipia_train_min], ignore_index=True).sample(
    frac=1.0, random_state=SEED
).reset_index(drop=True)
combined_val = pd.concat([eval_val, bipia_val_min], ignore_index=True).reset_index(drop=True)

print(f'Combined train: {len(combined_train)} rows')
print(f'  by dataset:\n{combined_train["dataset"].value_counts()}')
print(f'  class balance: {combined_train["label"].value_counts().to_dict()}')

model_b, tok_b, trainer_b, time_b = train_lora(
    train_df=combined_train, val_df=combined_val,
    run_label='v3b_combined_aug',
    adapter_save_path=ADAPTER_DIR_B,
    num_epochs=3, learning_rate=2e-4, batch_size=16,
)

In [ ]:
result_b_bipia = evaluate_on(trainer_b, tok_b, bipia_test, 'BIPIA-aug test')
result_b_evalset = evaluate_on(trainer_b, tok_b, eval_test, 'eval_set test')

metrics_b = [result_b_bipia['metrics'], result_b_evalset['metrics']]
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = eval_test[eval_test['dataset'] == ds].reset_index(drop=True)
    if len(sub) > 0:
        r = evaluate_on(trainer_b, tok_b, sub, f'  eval_set/{ds}')
        metrics_b.append(r['metrics'])

print_metrics_table(metrics_b, 'Variant B: combined (eval_set + BIPIA-aug) LoRA')

bipia_test_with_pred['pred_b'] = result_b_bipia['y_pred']
bipia_test_with_pred['score_b'] = result_b_bipia['y_score']

## 6. Robustness checks (same as NB10b §8)

Five checks: duplicates, length-shortcut, score-distribution degeneracy (the key one), confusion matrix, Variant B interference vs §5.11.

In [ ]:
# 6.1 Duplicates
for source_name, splits_df in [
    ('BIPIA-aug', bipia_splits),
    ('combined', pd.concat([
        combined_train.assign(split='train'),
        combined_val.assign(split='val'),
        eval_test.assign(split='test'),
        bipia_test.assign(split='test'),
    ], ignore_index=True)),
]:
    train_set = set(splits_df[splits_df['split'] == 'train']['prompt'])
    test_set = set(splits_df[splits_df['split'] == 'test']['prompt'])
    overlap = train_set & test_set
    pct = 100 * len(overlap) / max(len(test_set), 1)
    flag = 'CLEAN' if pct < 1.0 else ('CAVEAT' if pct < 5.0 else 'WARNING')
    print(f'{source_name:<10}: {len(overlap)} of {len(test_set)} test prompts overlap train ({pct:.2f}%)  [{flag}]')

In [ ]:
# 6.3 Score-distribution degeneracy check (THE key check vs NB10b)
# NB10b had Cohen d = 0.48 (A) and 0.13 (B), both degenerate.
# Target: Cohen d > 1.5 on BIPIA-aug test = the augmentation worked.

for variant_label, score_col in [('A (BIPIA-aug only)', 'score_a'), ('B (combined)', 'score_b')]:
    clean = bipia_test_with_pred[bipia_test_with_pred['label']==0][score_col]
    attack = bipia_test_with_pred[bipia_test_with_pred['label']==1][score_col]
    sep = abs(attack.mean() - clean.mean())
    pooled_std = ((clean.std()**2 + attack.std()**2) / 2) ** 0.5
    cohen_d = sep / max(pooled_std, 1e-6)
    print(f'Variant {variant_label} score distribution on BIPIA-aug test:')
    print(f'  clean  (n={len(clean):>3}): mean={clean.mean():.3f}  std={clean.std():.3f}  range=[{clean.min():.3f}, {clean.max():.3f}]')
    print(f'  attack (n={len(attack):>3}): mean={attack.mean():.3f}  std={attack.std():.3f}  range=[{attack.min():.3f}, {attack.max():.3f}]')
    print(f'  separation = {sep:.3f},  Cohen d = {cohen_d:.2f}')
    if cohen_d > 1.5:
        print(f'  HEALTHY: clean and attack distributions well separated. Augmentation worked.')
    elif cohen_d > 0.5:
        print(f'  MODEST: real signal but room to improve. Augmentation partially worked.')
    else:
        print(f'  DEGENERATE: still no signal (same as NB10b). Augmentation did not help.')
    print()

In [ ]:
# 6.4 Confusion matrices on BIPIA-aug test
for variant_label, pred_col in [('A', 'pred_a'), ('B', 'pred_b')]:
    cm = confusion_matrix(bipia_test_with_pred['label'], bipia_test_with_pred[pred_col], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / max(tn + fp, 1)
    fnr = fn / max(tp + fn, 1)
    print(f'Variant {variant_label}: TN={tn:>4} FP={fp:>4} FN={fn:>4} TP={tp:>4}   FPR={fpr:.3f} FNR={fnr:.3f}')

In [ ]:
# 6.5 Variant B interference check vs §5.11 LoRA-from-ProtectAI
LORA_V1_PER_DATASET = {'overall': 0.981, 'deepset': 0.967, 'neuralchemy': 0.985, 'spml': 0.986}
print(f'{"slice":<15} {"variant_B_F1":>13} {"§5.11_F1":>10} {"delta":>8}  verdict')
for m in metrics_b:
    label = m['label'].strip()
    if 'eval_set test' in label:
        key = 'overall'
    elif 'eval_set/' in label:
        key = label.split('/')[-1].strip()
    else:
        continue
    if key not in LORA_V1_PER_DATASET:
        continue
    delta = m['f1'] - LORA_V1_PER_DATASET[key]
    verdict = 'NO INTERFERENCE' if delta >= -0.01 else ('minor degradation' if delta >= -0.03 else 'INTERFERENCE')
    print(f'  {key:<13} {m["f1"]:>13.3f} {LORA_V1_PER_DATASET[key]:>10.3f} {delta:>+8.3f}  {verdict}')

## 7. Save and headline comparison vs NB10b

In [ ]:
def serializable(m):
    out = dict(m)
    out['precision_ci'] = list(m['precision_ci'])
    out['recall_ci'] = list(m['recall_ci'])
    return out

robustness = {}
for variant_label, score_col in [('a', 'score_a'), ('b', 'score_b')]:
    clean = bipia_test_with_pred[bipia_test_with_pred['label']==0][score_col]
    attack = bipia_test_with_pred[bipia_test_with_pred['label']==1][score_col]
    sep = float(abs(attack.mean() - clean.mean()))
    pooled_std = float(((clean.std()**2 + attack.std()**2) / 2) ** 0.5)
    cohen_d = sep / max(pooled_std, 1e-6)
    robustness[f'variant_{variant_label}_score_separation'] = {
        'clean_mean': float(clean.mean()), 'clean_std': float(clean.std()),
        'attack_mean': float(attack.mean()), 'attack_std': float(attack.std()),
        'separation': sep, 'cohen_d': cohen_d,
    }

summary = {
    'experiment': 'lora_v3_bipia_augmented_retraining',
    'base_model': BASE_MODEL,
    'augmentation': {
        'source_csv': 'bipia_email_qa_prompts_augmented.csv',
        'n_clean_total': int((bipia['label']==0).sum()),
        'n_clean_train': int((bipia_train['label']==0).sum()),
        'note': 'Each of 50 base emails paired with 5 generic legitimate questions; clean train rows: 35 (NB10b) -> ~210 (NB10c)',
    },
    'lora_config': {'r': 16, 'alpha': 32, 'dropout': 0.1, 'target_modules': 'all-linear'},
    'training': {
        'epochs': 3, 'lr': 2e-4, 'batch_size': 16, 'seed': SEED,
        'class_weighted_loss': True,
        'time_min_variant_a': time_a / 60,
        'time_min_variant_b': time_b / 60,
    },
    'variant_a_bipia_aug_only': {
        'train_size': len(bipia_train), 'val_size': len(bipia_val),
        'metrics': [serializable(m) for m in metrics_a],
    },
    'variant_b_combined_aug': {
        'train_size': len(combined_train), 'val_size': len(combined_val),
        'metrics': [serializable(m) for m in metrics_b],
    },
    'robustness_checks': robustness,
}

metrics_path = RESULTS_DIR / 'lora_v3_metrics.json'
metrics_path.write_text(json.dumps(summary, indent=2))
print(f'Saved {metrics_path}')

per_row_path = RESULTS_DIR / 'lora_v3_bipia_test_preds.csv'
bipia_test_with_pred[['row_id', 'attack_category', 'label',
                      'pred_a', 'score_a', 'pred_b', 'score_b']].to_csv(per_row_path, index=False)
print(f'Saved {per_row_path}')

print('\n=== Headline comparison vs NB10b ===')
print(f'NB10b Variant A BIPIA  Cohen d = 0.48  (DEGENERATE)')
print(f'NB10b Variant B BIPIA  Cohen d = 0.13  (DEGENERATE)')
print(f'NB10c Variant A BIPIA  Cohen d = {robustness["variant_a_score_separation"]["cohen_d"]:.2f}')
print(f'NB10c Variant B BIPIA  Cohen d = {robustness["variant_b_score_separation"]["cohen_d"]:.2f}')
print()
print(f'NB10c Variant A BIPIA  macro F1 = {metrics_a[0]["macro_f1"]:.3f}, bal_acc = {metrics_a[0]["balanced_accuracy"]:.3f}, MCC = {metrics_a[0]["mcc"]:.3f}')
print(f'NB10c Variant B BIPIA  macro F1 = {metrics_b[0]["macro_f1"]:.3f}, bal_acc = {metrics_b[0]["balanced_accuracy"]:.3f}, MCC = {metrics_b[0]["mcc"]:.3f}')
print(f'NB10c Variant B eval_set overall F1 = {metrics_b[1]["f1"]:.3f}  (vs §5.11 0.981)')

## 8. Interpretation guide

### Did the augmentation work?

| Variant A BIPIA Cohen d | Reading |
|---|---|
| > 1.5 | Augmentation worked. Defense A CAN learn BIPIA discrimination given sufficient clean controls. Data scarcity was the binding constraint. |
| 0.5 - 1.5 | Augmentation helped but not enough. Need more clean variety (different wrappers, different emails, or longer training). |
| < 0.5 | Augmentation did not help. The failure mode is architectural, not data-bound. Generic questions don't add enough semantic variety. |

### Did Variant B preserve direct injection?

Variant B eval_set F1 should stay ≈ 0.981 per §5.11. If it dropped to 0.95 or below, BIPIA-aug introduced training noise that hurt direct-injection performance. Pattern to watch: deepset is the most sensitive (smallest dataset).

### What to write in §5.11 if NB10c succeeded

Two-step finding:
1. NB10b: BIPIA's 50 native clean controls are insufficient for input-classifier training; the model latches onto format rather than content.
2. NB10c: pairing each base email with 5 generic legitimate questions (300 unique clean rows) fixes this; combined LoRA achieves Cohen d > 1.5 on BIPIA AND maintains §5.11 F1 on direct injection.

Deployment relevance: when fine-tuning Defense A for indirect injection, a 6:1 (or better) ratio of clean variation to attack variation is required. Companies deploying this have unlimited clean emails from their own corpus, so this is not a benchmark-curation artifact — it's a finding about training-set design.

### What to write in §5.11 if NB10c also failed

The failure is then architectural, not data-bound. Input-classifier discrimination of indirect injection at the prompt level may be intrinsically harder than direct injection (where lexical override patterns are the signal). Recommend Defense B (output judge with BIPIA addendum from §5.5b) as the right architectural layer for indirect injection. Stop here.

### Files to download to repo

- `MyDrive/capstone_lora/results/lora_v3_metrics.json` → `results/lora_v3_metrics.json`
- `MyDrive/capstone_lora/results/lora_v3_bipia_test_preds.csv` → `results/lora_v3_bipia_test_preds.csv`
- `MyDrive/capstone_lora/data/bipia_splits_augmented.parquet` → `results/bipia_splits_augmented.parquet`